# PyTorch dataset template

In [1]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import sys
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm
import matplotlib.gridspec as gridspec

import yaml

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

## Load configuration


In [2]:
# The config is the single source of truth. load_flare_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically in the notebook and in production.
#
# Typed means: cfg.data.channels instead of config["data"]["channels"] — you get IDE
# completion, and a misspelled key fails here with a list of the valid ones instead of
# silently doing nothing.
from downstream_apps.template.configs import load_flare_config

cfg = load_flare_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")

# Paths in the YAML are relative to the config file, and come back resolved:
print(f"  train index : {cfg.data.train_data_path}")
print(f"  scalers     : {cfg.data.scalers_path}")


## Download assets



In [3]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# This notebook only needs the scalers - no backbone is loaded.
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


## Define DS dataset

In [4]:
from downstream_apps.flare_forecast.datasets.flare_dataset import flare_Dataset

## Initialize class without Surya stacks


## Test length and structure



## Define dataloader


## Initialize class with Surya stacks



In [5]:
train_dataset = flare_Dataset(
    #### All these lines are required by the parent HelioNetCDFDataset class
    index_path=cfg.data.train_data_path,
    time_delta_input_minutes=cfg.data.time_delta_input_minutes,
    time_delta_target_minutes=cfg.data.time_delta_target_minutes,
    n_input_timestamps=cfg.model.time_embedding.time_dim,
    rollout_steps=cfg.rollout_steps,
    channels=cfg.data.channels,
    drop_hmi_probability=cfg.drop_hmi_probability,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
    scalers=scalers,
    phase="train",          # "val" disables random channel masking and flips
    # How s3:// paths in the index are read: "download" (default) | "simplecache" | "stream".
    # "download" fetches each file into s3_cache_dir first — recommended for NetCDF.
    s3_mode=cfg.data.s3_mode,
    s3_storage_options={"anon": cfg.data.s3_anon},
    s3_cache_dir=cfg.data.s3_cache_dir,
    #### Put your downstream (DS) specific parameters below this line
    return_surya_stack=True,
    max_number_of_samples=6,
    ds_flare_index_path=cfg.data.flare_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

# Notebooks 1 and 2 skip this block entirely: build_helio_dataloaders() in
# workshop_infrastructure/datasets/builders.py fills in everything above the
# "downstream specific" line straight from the config.


In [6]:
item = train_dataset.__getitem__(0)
item.keys()

In [7]:
item['ts'].shape

## Plotting input stack



In [8]:
unnormalized_ts = train_dataset.inverse_transform_data(item['ts'][:,0,...])
channel_order = cfg.data.channels
fig = plt.figure(figsize=np.array([4,4]), dpi=300)
gs = gridspec.GridSpec(4, 4, figure=fig, wspace=0, hspace=0)

limits = {}

for i in range(4):
    for j in range(4):
        n = i*4 + j
        if n < len(channel_order):

            ax = fig.add_subplot(gs[i,j])
            channel = channel_order[n]
            if 'hmi' not in channel:
                lim = np.percentile(unnormalized_ts[n,...][unnormalized_ts[n,...]!=0], 99)
                ax.imshow(unnormalized_ts[n,...], cmap=f'sdo{channel}', vmin=0, vmax=lim)
                font_color = 'w'

            else:
                font_color = 'k'
                if "_v" not in channel:
                    lim = 1000
                    ax.imshow(unnormalized_ts[n,...], cmap=f'hmimag', vmin = -lim, vmax=lim)
                else:
                    lim = 1000
                    ax.imshow(unnormalized_ts[n,...], cmap=f'coolwarm', vmin = -lim, vmax=lim)                    

            ax.text(0.01, 0.99, channel, transform=ax.transAxes, horizontalalignment='left', verticalalignment='top', color=font_color, fontsize=5)  
            ax.set_xticks([])
            ax.set_yticks([])             


In [9]:
data_loader = DataLoader(
                dataset=train_dataset,
                batch_size=2
            )

In [10]:
batch = next(iter(data_loader))
batch.keys()

In [11]:
batch['ts'].shape

In [12]:
#now plot next to each other:

flipped_ts = train_dataset.inverse_transform_data(item['ts'][:,0,...])
original_ts = np.flip(flipped_ts, axis=-2)
channel_order = cfg.data.channels
n = 0  # index into channel_order — pick which channel to test
channel = channel_order[n]

fig = plt.figure(figsize=np.array([8,4]), dpi=100)  # width doubled for 2 panels
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05, hspace=0)

for col, (plot_ts, title) in enumerate([(flipped_ts, "flipped"), (original_ts, "original")]):
    ax = fig.add_subplot(gs[0, col])
    if 'hmi' not in channel:
        lim = np.percentile(plot_ts[n,...][plot_ts[n,...]!=0], 99)
        ax.imshow(plot_ts[n,...], cmap=f'sdo{channel}', vmin=0, vmax=lim)
        font_color = 'w'
    else:
        font_color = 'k'
        lim = 1000
        if "_v" not in channel:
            ax.imshow(plot_ts[n,...], cmap='hmimag', vmin=-lim, vmax=lim)
        else:
            ax.imshow(plot_ts[n,...], cmap='coolwarm', vmin=-lim, vmax=lim)

    ax.text(0.01, 0.99, f"{channel} ({title})", transform=ax.transAxes,
            horizontalalignment='left', verticalalignment='top',
            color=font_color, fontsize=5)
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

In [ ]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import sys
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm
import matplotlib.gridspec as gridspec

import yaml

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

In [19]:
from downstream_apps.template.configs import load_flare_config

cfg = load_flare_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")

# Paths in the YAML are relative to the config file, and come back resolved:
print(f"  train index : {cfg.data.train_data_path}")
print(f"  scalers     : {cfg.data.scalers_path}")


In [20]:
train_dataset = flare_Dataset(
    #### All these lines are required by the parent HelioNetCDFDataset class
    index_path=cfg.data.train_data_path,
    time_delta_input_minutes=cfg.data.time_delta_input_minutes,
    time_delta_target_minutes=cfg.data.time_delta_target_minutes,
    n_input_timestamps=cfg.model.time_embedding.time_dim,
    rollout_steps=cfg.rollout_steps,
    channels=cfg.data.channels,
    drop_hmi_probability=cfg.drop_hmi_probability,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
    scalers=scalers,
    phase="train",          # "val" disables random channel masking and flips
    # How s3:// paths in the index are read: "download" (default) | "simplecache" | "stream".
    # "download" fetches each file into s3_cache_dir first — recommended for NetCDF.
    s3_mode=cfg.data.s3_mode,
    s3_storage_options={"anon": cfg.data.s3_anon},
    s3_cache_dir=cfg.data.s3_cache_dir,
    #### Put your downstream (DS) specific parameters below this line
    return_surya_stack=True,
    max_number_of_samples=6,
    ds_flare_index_path=cfg.data.flare_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

# Notebooks 1 and 2 skip this block entirely: build_helio_dataloaders() in
# workshop_infrastructure/datasets/builders.py fills in everything above the
# "downstream specific" line straight from the config.


In [17]:
item = train_dataset.__getitem__(0)
print(item['forecast'])       # scalar label, e.g. array(0.42, dtype=float32)
print(item['flipped'].shape)  # image stack, e.g. (13, 2, 4096, 4096)

In [18]:
print('ts' in item, 'flipped' in item)